In [1]:
# Install and load libraries
%pip install grad-cam
%matplotlib inline
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from tempfile import TemporaryDirectory
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets.folder import default_loader
import copy
import cv2
cudnn.benchmark = True
plt.ion()  # interactive mode
from pytorch_grad_cam import GradCAM, HiResCAM, ScoreCAM, GradCAMPlusPlus, AblationCAM, XGradCAM, EigenCAM, FullGrad
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 61.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 108.0 MB/s

In [2]:
# CNN models
class ModelWithNumericalFeatures(nn.Module):
    def __init__(self, base_model, num_classes):
        super(ModelWithNumericalFeatures, self).__init__()
        self.base_model = base_model
        self.model_type = type(base_model).__name__

        # Get the number of features from the base model
        if hasattr(base_model, 'fc'):  # ResNet
            num_features = base_model.fc.in_features
            base_model.fc = nn.Identity()
        elif hasattr(base_model, 'classifier'):  # ConvNeXt, EfficientNet, DenseNet
            # This part handles different structures for 'classifier'
            if isinstance(base_model.classifier, nn.Sequential):
                # Attempt to find the last Linear layer's in_features
                # For ConvNeXt, it's often classifier[2].in_features
                # For EfficientNet (some variants), it's classifier[-1].in_features
                # Defaulting to the last layer if it's linear
                if len(base_model.classifier) > 0 and isinstance(base_model.classifier[-1], nn.Linear):
                    num_features = base_model.classifier[-1].in_features
                elif len(base_model.classifier) > 2 and isinstance(base_model.classifier[2], nn.Linear): # Specific for ConvNeXt like structure
                    num_features = base_model.classifier[2].in_features
                else: # Fallback or needs specific handling for other Sequential classifiers
                    # You might need to inspect your specific model if this doesn't work
                    print(f"Warning: Could not automatically determine num_features for Sequential classifier in {self.model_type}. Defaulting to a common value or erroring soon.")
                    # Example: for convnext_tiny, classifier is (norm, flatten, linear, norm) -> classifier[2] is Linear
                    # Let's try to be more robust for models like ConvNeXt or EfficientNet
                    last_linear_layer = None
                    for layer in reversed(list(base_model.classifier.children())):
                        if isinstance(layer, nn.Linear):
                            last_linear_layer = layer
                            break
                    if last_linear_layer:
                        num_features = last_linear_layer.in_features
                    else:
                        raise ValueError(f"Cannot determine num_features for {self.model_type} with Sequential classifier.")

            elif isinstance(base_model.classifier, nn.Linear):  # DenseNet, some EfficientNet variants
                num_features = base_model.classifier.in_features
            else:
                raise ValueError(f"Unsupported classifier type in base_model: {type(base_model.classifier)}")
            base_model.classifier = nn.Identity() # Remove original classifier
        else:
            raise ValueError("Base model must have 'fc' or 'classifier' attribute.")

        # Embedding for 2 numerical features (DAS and DOY)
        self.numerical_features_embedding = nn.Sequential(
            nn.Linear(2, 64), # Input dimension is 2 (DAS, DOY)
            nn.ReLU(),
            nn.Linear(64, 256) # Output embedding size
        )

        # Classifier for combined features
        self.classifier = nn.Sequential(
            nn.Linear(num_features + 256, 512), # num_image_features + num_embedded_numerical_features
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x_image, x_numerical):
        # Extract features from the base model
        image_features = self.base_model(x_image)

        # Handle feature pooling if necessary (some models return pooled features, others don't)
        if image_features.dim() > 2: # e.g. (batch_size, channels, H, W)
            image_features = torch.mean(image_features, dim=[2, 3]) # Global average pooling

        # Process numerical features
        numerical_embedded_features = self.numerical_features_embedding(x_numerical)

        # Combine features
        combined_features = torch.cat((image_features, numerical_embedded_features), dim=1)

        # Final classification
        return self.classifier(combined_features)

In [4]:
# Load the pre-trained models
cnn_model = 'resnet50'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if cnn_model =='convnext_tiny':
    base_model = models.convnext_tiny(weights='IMAGENET1K_V1')
if cnn_model == 'densenet121':
    base_model = models.densenet121(weights='IMAGENET1K_V1')
if cnn_model =='efficientnet_b3':
    base_model = models.efficientnet_b3(weights='IMAGENET1K_V1')
if cnn_model =='resnet18':
    base_model = models.resnet18(weights='IMAGENET1K_V1')
if cnn_model == 'resnet50':
    base_model = models.resnet50(weights='IMAGENET1K_V2')
model = ModelWithNumericalFeatures(base_model, num_classes=3).to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 127MB/s]


In [5]:
# Load saved weights
cnn_model = 'resnet50'
model.load_state_dict(torch.load('/content/best_accuracy_model.pt'))
if cnn_model =='convnext_tiny':
  target_layer = model.base_model.features[-1]
if cnn_model == 'densenet121':
  target_layer = model.base_model.features.denseblock4.denselayer16.conv2
if cnn_model =='efficientnet_b3':
  target_layer = model.base_model.features[-1]
if cnn_model =='resnet18':
  target_layer = model.base_model.layer4[-1]
if cnn_model == 'resnet50':
  target_layer = model.base_model.layer4[-1]

In [ ]:
#target_layer = model.base_model.layer4[-1] # ResNet18, ResNet50
#target_layer = model.base_model.features[-1] # ConvNext tiny, EfficientNet B3
#target_layer = model.base_model.features.denseblock4.denselayer16.conv2 # DenseNet 121

In [6]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self.forward_hook)
        target_layer.register_full_backward_hook(self.full_backward_hook)

    def forward_hook(self, module, input, output):
        self.activations = output.detach()

    def full_backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def compute_heatmap(self, image_tensor, numerical_tensor, class_idx=None):
        # Forward pass
        logits = self.model(image_tensor, numerical_tensor)
        self.model.zero_grad()

        if class_idx is None:
            class_idx = torch.argmax(logits, dim=1).item()

        # Compute gradients for the target class
        one_hot_output = torch.zeros_like(logits)
        one_hot_output[0, class_idx] = 1
        logits.backward(gradient=one_hot_output)

        # Compute Grad-CAM heatmap
        weights = torch.mean(self.gradients, dim=[2, 3], keepdim=True)
        heatmap = torch.sum(weights * self.activations, dim=1, keepdim=True)
        heatmap = torch.relu(heatmap)
        heatmap /= torch.max(heatmap)

        # Get the predicted class probability
        probs = torch.softmax(logits, dim=1)
        predicted_prob = probs[0, class_idx].item()

        return heatmap.squeeze().cpu().numpy(), class_idx, predicted_prob

In [7]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [8]:
import matplotlib.pyplot as plt
import cv2

def save_heatmap(img_path: str, heatmap: np.ndarray, save_path: str, predicted_class_idx: int, predicted_prob: float) -> None:
    # Read the image
    img = cv2.imread(img_path)
    if img is None:
        print(f"Warning: Could not read image at {img_path}")
        return

    # Resize heatmap and apply colormap
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    # Superimpose heatmap on the original image
    superimposed_img = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)

    # Add prediction text to the image
    class_map = {0: 'basic', 1: 'moderate', 2: 'superior'}
    pred_text = f"Pred: {class_map.get(predicted_class_idx, 'Unknown')} ({predicted_prob:.2f})"
    cv2.putText(superimposed_img, pred_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

    # Save the image
    cv2.imwrite(save_path, superimposed_img)

In [9]:
from google.colab import drive
drive.mount('/gdrive', force_remount = True)
!sudo apt-get install unzip
!unzip -q '/gdrive/My Drive/pbi_kenya.zip' -d '/content' ## Executed in harold22010@gmail.com:: densenet121

Mounted at /gdrive
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unzip is already the newest version (6.0-26ubuntu3.2).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [10]:
# Load Configuration and Prepare Directories
import yaml
import os
from tqdm import tqdm # For a nice progress bar

# Load the YAML configuration
def load_config_yaml(yaml_file):
    with open(yaml_file, "r") as file:
        config_data = yaml.safe_load(file)
    return config_data

# --- Load paths from config.yaml ---
# Make sure your config.yaml is uploaded to the Colab environment
config_params = load_config_yaml("config.yaml")

dataset_path = config_params["dataset"]["in_path"]
output_base_path = config_params["dataset"]["out_path"]
cnn_model = config_params["model"]["name"]

# Define the main output path for the model's results
output_path = os.path.join(output_base_path, 'results_' + cnn_model)
# Define a specific sub-directory for the heatmaps
heatmap_output_dir = os.path.join(output_path, 'heatmaps')

# Create the directories if they don't exist
os.makedirs(output_path, exist_ok=True)
os.makedirs(heatmap_output_dir, exist_ok=True)

print(f"Dataset path: {dataset_path}")
print(f"Heatmaps will be saved in: {heatmap_output_dir}")

Dataset path: /content/pbi_kenya
Heatmaps will be saved in: /gdrive/My Drive/pbi_kenya/results_resnet50/heatmaps


In [11]:
# Use the validation set for generating heatmaps
# This assumes you have a 'val_features.csv' as used in pbi_categorical.py
val_csv_path = os.path.join(dataset_path, 'val_features.csv')
val_root_dir = os.path.join(dataset_path, 'val')

# Use the existing 'val' transform
val_transform = data_transforms['val']

# Define the custom dataset class (copied from your setup.py for clarity)
# If setup.py is in the same directory, you can just import it.
class ImageDatasetWithNumericalFeatures(Dataset):
    def __init__(self, csv_path, root_dir, transform=None):
        self.annotations = pd.read_csv(csv_path)
        self.root_dir = root_dir
        self.transform = transform
        self.loader = default_loader
        self.class_to_idx = {'basic': 0, 'moderate': 1, 'superior': 2}

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        relative_path = self.annotations.iloc[idx, 0]
        img_name = os.path.join(self.root_dir, relative_path)

        days_after_sowing = self.annotations.iloc[idx, 1]
        day_of_year = self.annotations.iloc[idx, 2]

        label_name = os.path.split(os.path.dirname(relative_path))[-1]
        label = self.class_to_idx[label_name]

        image = self.loader(img_name)
        if self.transform:
            image = self.transform(image)

        return {
            'image': image,
            'numerical_features': torch.tensor([days_after_sowing, day_of_year], dtype=torch.float32),
            'label': torch.tensor(label),
            'relative_path': relative_path # Keep the relative path for saving
        }


# Create the validation dataset
val_dataset = ImageDatasetWithNumericalFeatures(val_csv_path, val_root_dir, transform=val_transform)

# Create the DataLoader with batch_size=1
# Set shuffle=False to process in a predictable order
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)

print(f"Found {len(val_dataset)} images in the validation set.")

Found 6045 images in the validation set.


In [12]:
# Ensure the model is in evaluation mode
model.eval()

# Re-initialize the GradCAM instance (target_layer should already be defined)
gradcam = GradCAM(model, target_layer)

# Loop over the validation dataloader
print("Starting heatmap generation...")
for i, batch in tqdm(enumerate(val_loader), total=len(val_loader)):
    # Get data and move to device
    image_tensor = batch['image'].to(device)
    numerical_features = batch['numerical_features'].to(device)
    relative_path = batch['relative_path'][0] # Unpack from list

    # --- Generate Heatmap ---
    try:
        heatmap, pred_idx, pred_prob = gradcam.compute_heatmap(image_tensor, numerical_features)
    except Exception as e:
        print(f"Error processing {relative_path}: {e}")
        continue

    # --- Construct Paths and Save ---
    # Full path to the original image
    original_img_path = os.path.join(val_root_dir, relative_path)
    # Full path for the output heatmap image
    save_path_full = os.path.join(heatmap_output_dir, relative_path)

    # Create subdirectories if they don't exist (e.g., 'heatmaps/basic/')
    os.makedirs(os.path.dirname(save_path_full), exist_ok=True)

    # Save the heatmap
    save_heatmap(original_img_path, heatmap, save_path_full, pred_idx, pred_prob)

print(f"\nHeatmap generation complete. All files saved in {heatmap_output_dir}")

Starting heatmap generation...


  0%|          | 0/6045 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/autograd/graph.py:823: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:180.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
100%|██████████| 6045/6045 [05:16<00:00, 19.12it/s]


Heatmap generation complete. All files saved in /gdrive/My Drive/pbi_kenya/results_resnet50/heatmaps
